In [ ]:
# ============================================================
# GWP100-BASED LEAKAGE PENALTY AND RETAINED STORAGE METRIC
# ============================================================
# This block computes GWP100-based CO2-equivalent screening metrics.
#
# Metrics:
# - CO2_eq_leakage:
#   GWP100-based CO2-equivalent leakage penalty from leaked CO2 mass.
#
# - CO2_eq_retained:
#   Retained CO2-equivalent storage metric from stored CO2 mass.
#
# Notes:
# - This is not a full life-cycle assessment.
# - No avoided-emissions counterfactual is modeled.
# - Because CF_GWP100 for CO2 is 1, the leakage penalty is
#   numerically equal to leaked CO2 mass.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# SETTINGS
# ============================================================

CF_GWP100_CO2 = 1.0  # t CO2-eq per t CO2

LEAKED_COL = "mass_CO2_leakage"
STORED_COL = "mass_CO2_total"
LEAKAGE_FRACTION_COL = "leakage_fraction"


# ============================================================
# GWP100 METRIC CALCULATION
# ============================================================

def compute_gwp100_metrics(
    df,
    leaked_col="mass_CO2_leakage",
    stored_col="mass_CO2_total",
    cf_gwp100=1.0
):
    """
    Compute GWP100-based CO2-equivalent screening metrics.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe containing stored and leaked CO2 mass.
    leaked_col : str
        Column name for leaked CO2 mass.
    stored_col : str
        Column name for stored CO2 mass.
    cf_gwp100 : float
        GWP100 characterization factor for CO2.

    Returns
    -------
    pandas.DataFrame
        Dataframe with added GWP100-based metric columns.

    Notes
    -----
    This calculation is a GWP100-based climate-impact characterization
    step and not a full life-cycle assessment.
    """

    df_out = df.copy()

    required_cols = [leaked_col, stored_col]
    missing_cols = [col for col in required_cols if col not in df_out.columns]

    if missing_cols:
        raise ValueError(
            f"Missing required columns for GWP100 metrics: {missing_cols}"
        )

    df_out["CO2_eq_leakage"] = df_out[leaked_col] * cf_gwp100
    df_out["CO2_eq_retained"] = df_out[stored_col] * cf_gwp100

    return df_out


df_gwp = df.copy().reset_index(drop=True)

df_gwp = compute_gwp100_metrics(
    df=df_gwp,
    leaked_col=LEAKED_COL,
    stored_col=STORED_COL,
    cf_gwp100=CF_GWP100_CO2
)

print("\n=== GWP100-BASED METRICS ADDED ===")
print(["CO2_eq_leakage", "CO2_eq_retained"])

if "leak_category" in df_gwp.columns:
    print("\nClass distribution used for GWP100 characterization:")
    print(df_gwp["leak_category"].value_counts())


# ============================================================
# SUMMARY STATISTICS
# ============================================================

summary_vars = [
    STORED_COL,
    LEAKED_COL,
    LEAKAGE_FRACTION_COL,
    "CO2_eq_retained",
    "CO2_eq_leakage",
]

available_summary_vars = [var for var in summary_vars if var in df_gwp.columns]

summary_stats = df_gwp[available_summary_vars].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
).T

summary_stats = summary_stats.rename(index={
    STORED_COL: "Stored CO2 mass",
    LEAKED_COL: "Leaked CO2 mass",
    LEAKAGE_FRACTION_COL: "Leakage fraction",
    "CO2_eq_retained": "Retained CO2-equivalent storage metric",
    "CO2_eq_leakage": "GWP100-based leakage penalty",
})

summary_stats_display = summary_stats.copy()
summary_stats_display = summary_stats_display.round(4)

print("\n=== GWP100-BASED SUMMARY STATISTICS ===")
print(summary_stats_display)

summary_stats.to_csv(
    TABLES_DIR / "Appendix_A3_GWP100_MetricSummary_exact.csv",
    index=True
)

summary_stats_display.to_csv(
    TABLES_DIR / "Appendix_A3_GWP100_MetricSummary_rounded.csv",
    index=True
)


# ============================================================
# KEY VALUES FOR MANUSCRIPT
# ============================================================

leak_penalty = df_gwp["CO2_eq_leakage"].replace([np.inf, -np.inf], np.nan).dropna()
retained_metric = df_gwp["CO2_eq_retained"].replace([np.inf, -np.inf], np.nan).dropna()

leak_min = leak_penalty.min()
leak_max = leak_penalty.max()
leak_mean = leak_penalty.mean()
leak_median = leak_penalty.median()
leak_p75 = leak_penalty.quantile(0.75)
leak_p90 = leak_penalty.quantile(0.90)

ret_p10 = retained_metric.quantile(0.10)
ret_p25 = retained_metric.quantile(0.25)
ret_median = retained_metric.median()
ret_p75 = retained_metric.quantile(0.75)
ret_p90 = retained_metric.quantile(0.90)

print("\n=== KEY VALUES FOR MANUSCRIPT ===")
print(f"GWP100-based leakage penalty range: {leak_min:.1f} to {leak_max:.1f} t CO2-eq")
print(f"GWP100-based leakage penalty mean: {leak_mean:.1f} t CO2-eq")
print(f"GWP100-based leakage penalty median: {leak_median:.1f} t CO2-eq")
print(f"GWP100-based leakage penalty P75: {leak_p75:.1f} t CO2-eq")
print(f"GWP100-based leakage penalty P90: {leak_p90:.1f} t CO2-eq")

print("\nRetained CO2-equivalent storage metric:")
print(f"P10: {ret_p10 / 1e6:.2f} Mt CO2-eq")
print(f"P25: {ret_p25 / 1e6:.2f} Mt CO2-eq")
print(f"P50 / median: {ret_median / 1e6:.2f} Mt CO2-eq")
print(f"P75: {ret_p75 / 1e6:.2f} Mt CO2-eq")
print(f"P90: {ret_p90 / 1e6:.2f} Mt CO2-eq")


key_values = pd.DataFrame({
    "Metric": [
        "GWP100-based leakage penalty min",
        "GWP100-based leakage penalty max",
        "GWP100-based leakage penalty mean",
        "GWP100-based leakage penalty median",
        "GWP100-based leakage penalty P75",
        "GWP100-based leakage penalty P90",
        "Retained CO2-equivalent storage metric P10",
        "Retained CO2-equivalent storage metric P25",
        "Retained CO2-equivalent storage metric median",
        "Retained CO2-equivalent storage metric P75",
        "Retained CO2-equivalent storage metric P90",
    ],
    "Value": [
        leak_min,
        leak_max,
        leak_mean,
        leak_median,
        leak_p75,
        leak_p90,
        ret_p10,
        ret_p25,
        ret_median,
        ret_p75,
        ret_p90,
    ],
    "Unit": [
        "t CO2-eq",
        "t CO2-eq",
        "t CO2-eq",
        "t CO2-eq",
        "t CO2-eq",
        "t CO2-eq",
        "t CO2-eq",
        "t CO2-eq",
        "t CO2-eq",
        "t CO2-eq",
        "t CO2-eq",
    ]
})

key_values.to_csv(
    TABLES_DIR / "GWP100_KeyValues_forManuscript.csv",
    index=False
)


# ============================================================
# FIGURE: LEAKAGE FRACTION VS GWP100-BASED LEAKAGE PENALTY
# ============================================================

x = df_gwp[LEAKAGE_FRACTION_COL].values
y = df_gwp["CO2_eq_leakage"].values

mask = np.isfinite(x) & np.isfinite(y)
x = x[mask]
y = y[mask]

# Linear trend for visualization only.
m, b = np.polyfit(x, y, 1)
x_sorted = np.sort(x)

fig, ax = plt.subplots(figsize=(8, 5), dpi=600)

hb = ax.hexbin(
    x,
    y,
    gridsize=55,
    cmap="viridis",
    bins="log",
    mincnt=1,
    linewidths=0.2,
    alpha=0.95
)

cbar = fig.colorbar(hb, ax=ax)
cbar.set_label("log(count)", fontsize=11)
cbar.ax.tick_params(labelsize=10)

ax.plot(
    x_sorted,
    m * x_sorted + b,
    color="#004D40",
    linewidth=2.5,
    label="Linear trend"
)

ax.set_xlabel("Leakage fraction (-)", fontsize=13)
ax.set_ylabel("GWP100-based leakage penalty (t CO₂-eq)", fontsize=13)
ax.set_title("Leakage Fraction vs GWP100-Based Leakage Penalty", fontsize=15)

legend = ax.legend(
    frameon=True,
    fancybox=True,
    framealpha=0.90,
    edgecolor="black",
    fontsize=11
)

legend.get_frame().set_linewidth(0.8)
legend.get_frame().set_facecolor("white")

ax.grid(alpha=0.25, linestyle="--", linewidth=0.5)

plt.tight_layout()

fig_path = FIGURES_DIR / "Figure_11_GWP100_LeakageFraction_vs_LeakagePenalty.png"
fig.savefig(fig_path, dpi=600, bbox_inches="tight")

plt.show()
plt.close(fig)

print(f"Saved: {fig_path}")


# ============================================================
# FIGURE: RETAINED CO2-EQUIVALENT STORAGE METRIC
# ============================================================

data = retained_metric.copy()
data_plot = data / 1e6  # Convert t to Mt.

p10 = np.percentile(data_plot, 10)
p50 = np.percentile(data_plot, 50)
p90 = np.percentile(data_plot, 90)

fig, ax = plt.subplots(figsize=(9, 5.8), dpi=600)

ax.hist(
    data_plot,
    bins=40,
    alpha=0.85,
    edgecolor="black"
)

ax.axvline(
    p10,
    linestyle="--",
    linewidth=2,
    label=f"P10 = {p10:.2f} Mt"
)

ax.axvline(
    p50,
    linestyle="-",
    linewidth=2,
    label=f"P50 = {p50:.2f} Mt"
)

ax.axvline(
    p90,
    linestyle=":",
    linewidth=2.5,
    label=f"P90 = {p90:.2f} Mt"
)

ax.set_xlabel("Retained CO₂-equivalent storage metric (Mt CO₂-eq)", fontsize=16)
ax.set_ylabel("Frequency", fontsize=16)
ax.set_title("Distribution of Retained CO₂-Equivalent Storage Metric", fontsize=18)

ax.tick_params(axis="both", labelsize=14)
ax.grid(alpha=0.2)

legend = ax.legend(
    frameon=True,
    fancybox=True,
    framealpha=0.90,
    edgecolor="black",
    loc="upper right",
    fontsize=13
)

legend.get_frame().set_linewidth(0.8)
legend.get_frame().set_facecolor("white")

plt.tight_layout()

fig_path = FIGURES_DIR / "Figure_12_GWP100_RetainedCO2eqStorageMetric.png"
fig.savefig(fig_path, dpi=600, bbox_inches="tight")

plt.show()
plt.close(fig)

print(f"Saved: {fig_path}")


# ============================================================
# OPTIONAL APPENDIX HISTOGRAMS
# ============================================================

appendix_histograms = {
    "CO2_eq_leakage": {
        "xlabel": "GWP100-based leakage penalty (Mt CO₂-eq)",
        "title": "Distribution of GWP100-Based Leakage Penalty",
        "filename": "Appendix_A3_Hist_GWP100_LeakagePenalty.png",
        "convert_to_Mt": True
    },
    "CO2_eq_retained": {
        "xlabel": "Retained CO₂-equivalent storage metric (Mt CO₂-eq)",
        "title": "Distribution of Retained CO₂-Equivalent Storage Metric",
        "filename": "Appendix_A3_Hist_RetainedCO2eqStorageMetric.png",
        "convert_to_Mt": True
    },
    LEAKAGE_FRACTION_COL: {
        "xlabel": "Leakage fraction (-)",
        "title": "Distribution of Leakage Fraction",
        "filename": "Appendix_A3_Hist_LeakageFraction.png",
        "convert_to_Mt": False
    }
}

for var, info in appendix_histograms.items():

    data = df_gwp[var].replace([np.inf, -np.inf], np.nan).dropna()

    p10 = np.percentile(data, 10)
    p50 = np.percentile(data, 50)
    p90 = np.percentile(data, 90)

    if info["convert_to_Mt"]:
        data_plot = data / 1e6
        p10_plot = p10 / 1e6
        p50_plot = p50 / 1e6
        p90_plot = p90 / 1e6
        lab_p10 = f"P10 = {p10_plot:.2f} Mt"
        lab_p50 = f"P50 = {p50_plot:.2f} Mt"
        lab_p90 = f"P90 = {p90_plot:.2f} Mt"
    else:
        data_plot = data
        p10_plot = p10
        p50_plot = p50
        p90_plot = p90
        lab_p10 = f"P10 = {p10_plot:.3f}"
        lab_p50 = f"P50 = {p50_plot:.3f}"
        lab_p90 = f"P90 = {p90_plot:.3f}"

    fig, ax = plt.subplots(figsize=(9, 5.8), dpi=600)

    ax.hist(
        data_plot,
        bins=40,
        alpha=0.85,
        edgecolor="black"
    )

    ax.axvline(
        p10_plot,
        color="#1f77b4",
        linestyle="--",
        linewidth=2.2,
        label=lab_p10
    )

    ax.axvline(
        p50_plot,
        color="#d62728",
        linestyle="-",
        linewidth=2.4,
        label=lab_p50
    )

    ax.axvline(
        p90_plot,
        color="#2ca02c",
        linestyle=":",
        linewidth=2.8,
        label=lab_p90
    )

    ax.set_xlabel(info["xlabel"], fontsize=16)
    ax.set_ylabel("Frequency", fontsize=16)
    ax.set_title(info["title"], fontsize=18)

    ax.tick_params(axis="both", labelsize=14)
    ax.grid(alpha=0.2)

    legend = ax.legend(
        frameon=True,
        fancybox=True,
        framealpha=0.90,
        edgecolor="black",
        loc="upper right",
        fontsize=13
    )

    legend.get_frame().set_linewidth(0.8)
    legend.get_frame().set_facecolor("white")

    plt.tight_layout()

    fig_path = FIGURES_DIR / info["filename"]
    fig.savefig(fig_path, dpi=600, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    print(f"Saved: {fig_path}")


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\nGWP100-based metric calculation completed successfully.")
print("Reminder: These metrics are screening indicators, not a full LCA.")
print("No avoided-emissions counterfactual is modeled.")